# Flight Delay Prediction

Flight disruptions cause substantial economic losses for airlines and significant inconvenience for travelers. In this project, we build a data mining pipeline that predicts domestic flight delays from historical flight records and weather forecast data.

To mitigate data sparsity across the U.S. aviation network, we restrict our scope to domestic flights between the **15 busiest U.S. airport hubs**.

The problem is framed in two ways:
- **Primary task — binary classification:** predict whether a flight arrives more than 15 minutes late.
- **Secondary task — regression:** predict the arrival delay in minutes.

To prevent data leakage, every predictive feature is limited to information available **2 hours before the scheduled departure (T − 2h)**. We use a chronological train/test split to mirror real-world forecasting, and compare a route-average baseline, symbolic models (Decision Trees, Random Forests) and non-symbolic models (SVM, XGBoost) using F1-score, ROC-AUC and MAE.

## Project structure

### 1: Data selection
- Choosing the subset of images that belong to our scope (top 15 U.S airport hubs).

### 2: Data preprocessing
- Missing Value Imputation
- Categorical Encoding
- Feature Normalization
- Class Imbalance Handling

### 3: Transformation
// TODO

### 4: Data Mining
// TODO

### 5: Interpretation and evaluation


## 1. Data selection

In [19]:
import pandas as pd

'''
    Since the dataframe has so many unwanted attributes, we need to choose which collumns or attributues we are interested in, the others will not go into
    the dataframe.
    Also, the last collumns have a great ammount of missing values, so indeed the best option is to drop them.
'''

COLS = [
    # time
    "FlightDate", "Month", "DayofMonth", "DayOfWeek",
    # flight / aircraft
    "Reporting_Airline", "Tail_Number", "Flight_Number_Reporting_Airline",
    # route
    "Origin", "Dest", "Distance",
    # schedule (known before departure)
    "CRSDepTime", "CRSArrTime", "CRSElapsedTime",
    # actual times (needed to build the incoming-aircraft feature, NOT as inputs)
    "DepTime", "ArrTime",
    # labels
    "ArrDelay", "ArrDelayMinutes", "ArrDel15",
    # filtering
    "Cancelled", "Diverted",
    # optional, analysis only
    "CarrierDelay", "WeatherDelay", "NASDelay", "SecurityDelay", "LateAircraftDelay",
]

# FINDING TOP 15 BUSIEST AIRPORTS
df_all = pd.read_csv("data/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2026_1.csv",
                     usecols=["Origin", "Dest"])

counts = df_all["Origin"].value_counts()

HUBS = counts.head(15).index.tolist()


# Final dataframe we will use
df = pd.read_csv("data/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2026_1.csv", usecols=COLS,
                 dtype={"CRSDepTime": str, "CRSArrTime": str})

# Filtering
df = df[df["Origin"].isin(HUBS) & df["Dest"].isin(HUBS)]
df = df[(df["Cancelled"] == 0) & (df["Diverted"] == 0)]  # No cancelled or diverted flights (desviados)

# Convert scheduled times from HHMM strings to numeric hour features.
df["scheduled_departure_hour"] = (
    pd.to_numeric(df["CRSDepTime"], errors="coerce") // 100
)
df["scheduled_arrival_hour"] = (
    pd.to_numeric(df["CRSArrTime"], errors="coerce") // 100
)

# Showing 10 examples
pd.set_option("display.max_columns", None) 
pd.set_option("display.width", 200)

df.sample(10, random_state=42)

,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,Tail_Number,Flight_Number_Reporting_Airline,Origin,Dest,CRSDepTime,DepTime,CRSArrTime,ArrTime,ArrDelay,ArrDelayMinutes,ArrDel15,Cancelled,Diverted,CRSElapsedTime,Distance,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
538822,1,7,3,2026-01-07,UA,N24514,2430,DEN,ORD,0750,739.0,1134,1104.0,-30.0,0.0,0.0,0.0,0.0,164.0,888.0,NaN,NaN,NaN,NaN,NaN
87472,1,29,4,2026-01-29,AS,N812AK,637,PHX,SEA,1100,1056.0,1319,1320.0,1.0,1.0,0.0,0.0,0.0,199.0,1107.0,NaN,NaN,NaN,NaN,NaN
329272,1,14,3,2026-01-14,WN,N456WN,528,LAS,PHX,1830,1824.0,2045,2025.0,-20.0,0.0,0.0,0.0,0.0,75.0,255.0,NaN,NaN,NaN,NaN,NaN
79926,1,5,1,2026-01-05,AS,N979AK,407,ORD,SEA,1533,1530.0,1816,1803.0,-13.0,0.0,0.0,0.0,0.0,283.0,1721.0,NaN,NaN,NaN,NaN,NaN
9516,1,18,7,2026-01-18,AA,N458AL,1925,MIA,DFW,1202,1154.0,1433,1437.0,4.0,4.0,0.0,0.0,0.0,211.0,1121.0,NaN,NaN,NaN,NaN,NaN
368678,1,22,4,2026-01-22,YX,N104HQ,4429,ATL,DCA,1404,1354.0,1559,1538.0,-21.0,0.0,0.0,0.0,0.0,115.0,547.0,NaN,NaN,NaN,NaN,NaN
125999,1,21,3,2026-01-21,DL,N509DT,450,MIA,LAX,0750,748.0,1045,1034.0,-11.0,0.0,0.0,0.0,0.0,355.0,2342.0,NaN,NaN,NaN,NaN,NaN
64433,1,23,5,2026-01-23,AA,N150UW,1722,CLT,MCO,1120,1116.0,1312,1257.0,-15.0,0.0,0.0,0.0,0.0,112.0,468.0,NaN,NaN,NaN,NaN,NaN
225503,1,30,5,2026-01-30,MQ,N225NN,3359,ORD,BOS,1152,1217.0,1517,1633.0,76.0,76.0,1.0,0.0,0.0,145.0,867.0,0.0,25.0,51.0,0.0,0.0
209955,1,2,5,2026-01-02,F9,N644FR,3142,ORD,CLT,0545,707.0,0852,950.0,58.0,58.0,1.0,0.0,0.0,127.0,599.0,58.0,0.0,0.0,0.0,0.0


In [20]:
print(f"After the data selection, we have {df.shape[0]} usable examples")

print(f"Top 15 busiest airports: {HUBS}")

After the data selection, we have 70627 usable examples
Top 15 busiest airports: ['ORD', 'ATL', 'DEN', 'DFW', 'PHX', 'LAX', 'CLT', 'MCO', 'LAS', 'SEA', 'DCA', 'SFO', 'LGA', 'MIA', 'BOS']


## 2. Data preprocessing

What the data selection step left us with (profiled before writing this section):

| Finding | Consequence |
|---|---|
| `ArrDelay` / `ArrDel15` are missing for every cancelled or diverted flight | No label exists, so these rows are **dropped**, not imputed. Step 1 already does this; the cleaning cell below re-checks it defensively |
| `CarrierDelay`, `WeatherDelay`, `NASDelay`, `SecurityDelay`, `LateAircraftDelay` are ~79% missing (only filled for delayed flights) | They are only known *after* the flight, so they are **leakage** and are excluded from the features (analysis only) |
| `Tail_Number` has a few hundred missing values and ~4,800 distinct values | Too high-cardinality to one-hot; handled in Problem 3 (aircraft type). Not used as a feature here |
| Data covers a single month (January 2026) | `Month` is constant; the chronological split is done within the month |
| Positive class (`ArrDel15 = 1`) is ~20% of the training flights | Mild imbalance, handled with SMOTE inside the training splits (as in the proposal) |
| Delay rate is ~20% in train but ~31% in test (the last week of January) | The chronological split exposes a distribution shift; worth investigating (see 2.5) |

# TODO: Weather data still not fetched and trated

### 2.1 Missing value handling
The flights with no delay have NaN attributes, so we want to fill those with 0, filtering out every tuple with NaN in the delay times would be a critical mistake, there are also some missing values in Tail_Number, but we do not care, we want to keep unknown aircrafts, since our prediction is based on Hubs and not aerolines themselves.

In [21]:
import numpy as np

print("Missing values before cleaning:")
print(df.isna().sum()[lambda s: s > 0], end="\n\n")

# 1. Rows with no label: cancelled / diverted flights never get a (meaningful) arrival delay
# This is expected to drop 0 rows since it is treated in data selection, but I write it here for order.
n_before = len(df)
df = df[(df["Cancelled"] == 0) & (df["Diverted"] == 0)].copy()
df = df.dropna(subset=["ArrDelay", "ArrDel15"])
print(f"Dropped {n_before - len(df)} cancelled/diverted/unlabelled flights -> {len(df)} remain")

# 2. Delay-cause columns: NaN just means "no delay was attributed" for on-time flights.
#    We fill with 0 so they are usable for analysis, but they are NOT model inputs (leakage).
CAUSE_COLS = ["CarrierDelay", "WeatherDelay", "NASDelay", "SecurityDelay", "LateAircraftDelay"]

# With this print we can see that the flights with no delay have NaN
df[CAUSE_COLS].sample(10,random_state=42)

Missing values before cleaning:
CarrierDelay         54821
WeatherDelay         54821
NASDelay             54821
SecurityDelay        54821
LateAircraftDelay    54821
dtype: int64

Dropped 0 cancelled/diverted/unlabelled flights -> 70627 remain


,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
538822,NaN,NaN,NaN,NaN,NaN
87472,NaN,NaN,NaN,NaN,NaN
329272,NaN,NaN,NaN,NaN,NaN
79926,NaN,NaN,NaN,NaN,NaN
9516,NaN,NaN,NaN,NaN,NaN
368678,NaN,NaN,NaN,NaN,NaN
125999,NaN,NaN,NaN,NaN,NaN
64433,NaN,NaN,NaN,NaN,NaN
225503,0.0,25.0,51.0,0.0,0.0
209955,58.0,0.0,0.0,0.0,0.0


In [22]:
df[CAUSE_COLS] = df[CAUSE_COLS].fillna(0)

# 3. Tail number: unknown aircraft are kept (the flight itself is valid) under an explicit label.
df["Tail_Number"] = df["Tail_Number"].fillna("UNKNOWN")

print("\nMissing values after cleaning:")
print(df.isna().sum().sum(), "remaining")


Missing values after cleaning:
0 remaining


In [23]:
# Build a proper timestamp from FlightDate + CRSDepTime (HHMM string). It is used for the
# chronological split. CRSDepTime can be "2400", which pandas cannot parse as a time, so we
# convert hours/minutes arithmetically instead.
dep_minutes = df["CRSDepTime"].str[:2].astype(int) * 60 + df["CRSDepTime"].str[2:].astype(int)
df["CRSDepMinutes"] = dep_minutes          # CRSDepTime as minutes since midnight (numeric)
df["SchedDep"] = pd.to_datetime(df["FlightDate"]) + pd.to_timedelta(dep_minutes, unit="m")

df["ArrDel15"] = df["ArrDel15"].astype(int)
df = df.sort_values("SchedDep").reset_index(drop=True)
print(df["SchedDep"].min(), "->", df["SchedDep"].max())

2026-01-01 00:14:00 -> 2026-01-31 23:59:00


### 2.2 Chronological train/test split

The split comes **before** any encoder, scaler or class weight is fitted, so that nothing is learned from test data. The first 80% of flights (by scheduled departure) are the training set; the last 20% are the test set. The test set keeps the natural class and route distribution, since it should reflect the real prediction scenario.

#### Why a chronological split and not a random one?
We split chronologically so the evaluation matches how the model would actually be used: trained on past flights, then asked to predict future ones.

In [24]:
cutoff = df["SchedDep"].quantile(0.80)

train = df[df["SchedDep"] <  cutoff].copy()
test  = df[df["SchedDep"] >= cutoff].copy()

print(f"Cutoff: {cutoff}")
print(f"Train: {len(train):>6}  ({train['SchedDep'].min().date()} -> {train['SchedDep'].max().date()})")
print(f"Test : {len(test):>6}  ({test['SchedDep'].min().date()} -> {test['SchedDep'].max().date()})")
print(f"Delayed share  train: {train['ArrDel15'].mean():.3f}   test: {test['ArrDel15'].mean():.3f}")

Cutoff: 2026-01-24 16:10:00
Train:  56498  (2026-01-01 -> 2026-01-24)
Test :  14129  (2026-01-24 -> 2026-01-31)
Delayed share  train: 0.202   test: 0.310


### 2.2b Time-series cross-validation on the training set

The proposal (section 4.1) rules out random K-fold, because shuffling lets a model train on flights that happen *after* the ones it is validated on. Tuning is instead done with **time-series cross-validation** (`TimeSeriesSplit`): each fold trains on an expanding window of earlier flights and validates on the block that follows. It is later passed as `cv=` to `GridSearchCV`. The test set is never touched here.

In [25]:
from sklearn.model_selection import TimeSeriesSplit

N_SPLITS = 5
tscv = TimeSeriesSplit(n_splits=N_SPLITS)     # train is already sorted by SchedDep

print(f"{'fold':<5}{'train n':>9}{'val n':>8}   {'train ends':<18}{'val period':<40}{'val delayed':>11}")
for k, (tr, va) in enumerate(tscv.split(train), 1):
    tr_df, va_df = train.iloc[tr], train.iloc[va]
    assert tr_df["SchedDep"].max() <= va_df["SchedDep"].min()   # no look-ahead
    print(f"{k:<5}{len(tr):>9}{len(va):>8}   {str(tr_df['SchedDep'].max().date()):<18}"
          f"{str(va_df['SchedDep'].min().date())} -> {str(va_df['SchedDep'].max().date()):<22}{va_df['ArrDel15'].mean():>11.3f}")

fold   train n   val n   train ends        val period                              val delayed
1         9418    9416   2026-01-04        2026-01-04 -> 2026-01-08                  0.213
2        18834    9416   2026-01-08        2026-01-08 -> 2026-01-12                  0.162
3        28250    9416   2026-01-12        2026-01-12 -> 2026-01-16                  0.144
4        37666    9416   2026-01-16        2026-01-16 -> 2026-01-20                  0.230
5        47082    9416   2026-01-20        2026-01-20 -> 2026-01-24                  0.146


### 2.3 Categorical encoding and 2.4 Feature normalization

- **One-hot encoding** for the low-cardinality categoricals: airline (12), origin (15), destination (15). `handle_unknown="ignore"` makes unseen categories encode as all zeros instead of crashing.
- **Standardisation** (zero mean, unit variance) for numeric features: `Distance`, `CRSElapsedTime` and `CRSDepTime` (as minutes since midnight), following the proposal. This matters for the SVM; tree-based models (Decision Tree, Random Forest, XGBoost) are unaffected by it.
- Both are fitted **on the training set only** and then applied to the test set.
- Not encoded here: `Tail_Number` and `Flight_Number_Reporting_Airline` (high-cardinality, see Problem 3) and the origin–destination route (see Problem 4). Further temporal features (weekday, hour, holiday proximity) are created in step 3 (Transformation).

In [26]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

CATEGORICAL_COLS = ["Reporting_Airline", "Origin", "Dest"]
NUMERIC_COLS     = ["Distance", "CRSElapsedTime", "CRSDepMinutes"]
# WEATHER_COLS = [...]   # TODO: weather features will be added here (Problem 1)
# NUMERIC_COLS += WEATHER_COLS

FEATURE_COLS = CATEGORICAL_COLS + NUMERIC_COLS

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_COLS),
        ("num", StandardScaler(), NUMERIC_COLS),
    ]
)

# Fit on TRAIN only, then transform both sets.
X_train = preprocessor.fit_transform(train[FEATURE_COLS])
X_test  = preprocessor.transform(test[FEATURE_COLS])
feature_names = preprocessor.get_feature_names_out()

y_train_cls, y_test_cls = train["ArrDel15"].values, test["ArrDel15"].values   # classification target
y_train_reg, y_test_reg = train["ArrDelay"].values,  test["ArrDelay"].values  # regression target

print(f"X_train: {X_train.shape}   X_test: {X_test.shape}")
print(f"Missing values in X: {np.isnan(X_train).sum() + np.isnan(X_test).sum()}")
print(f"Numeric check (train) mean ~0, std ~1:",
      X_train[:, -len(NUMERIC_COLS):].mean(axis=0).round(3),
      X_train[:, -len(NUMERIC_COLS):].std(axis=0).round(3))

X_train: (56498, 45)   X_test: (14129, 45)
Missing values in X: 0
Numeric check (train) mean ~0, std ~1: [ 0. -0.  0.] [1. 1. 1.]


### 2.5 Class imbalance handling (SMOTE)

Following the proposal (section 3.1), the delayed class (~1 in 5 training flights) is oversampled with **SMOTE**: synthetic delayed flights are interpolated between real, neighbouring delayed flights until both classes are the same size.

Rules that prevent leakage:
- SMOTE is applied **only to training data**. The hold-out test set and each CV validation block keep their natural class distribution.
- It runs **inside each time-series CV fold**, after that fold's preprocessor has been fitted. `imblearn.pipeline.Pipeline` does this automatically: resampling steps are skipped at prediction time.

Notes:
## TODO: Review this first point, and review proposal
- The proposal also mentions *stratified sampling*. A time-ordered split cannot be stratified without breaking chronology, so the class balance is restored by SMOTE instead of by stratifying the folds.
- SMOTE is applied after one-hot encoding, so synthetic rows contain fractional one-hot values (e.g. 0.4 for one airport). Tree models tolerate this; it is worth keeping in mind when interpreting results.
- We evaluate with **F1 and ROC-AUC** rather than accuracy: always predicting "on time" already scores ~69% accuracy on the test set.

**Distribution shift to keep in mind:** the test period has a much higher delay rate (~31%) than the training period (~20%). It is a real feature of forecasting forward in time, so the test set is not rebalanced. Calibration and the decision threshold should be examined in the evaluation step, and it is worth checking what drove the last week (e.g. a weather event, which the future weather features may explain).

In [27]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Reusable pipeline: preprocessing -> SMOTE -> model. Step 4 only has to swap in the model, e.g.
#   pipe = ImbPipeline([("prep", preprocessor), ("smote", SMOTE(random_state=42)), ("clf", model)])
#   GridSearchCV(pipe, param_grid, cv=tscv, scoring="f1")
resampler = ImbPipeline([
    ("prep",  preprocessor),
    ("smote", SMOTE(random_state=42)),
])

print(f"{'fold':<5}{'train before':>22}{'train after SMOTE':>24}{'val (untouched)':>22}")
for k, (tr, va) in enumerate(tscv.split(train), 1):
    X_tr, y_tr = resampler.fit_resample(train.iloc[tr][FEATURE_COLS], y_train_cls[tr])
    y_va = y_train_cls[va]
    b = np.bincount(y_train_cls[tr]); a = np.bincount(y_tr)
    print(f"{k:<5}{f'{b[0]} / {b[1]}':>22}{f'{a[0]} / {a[1]}':>24}{f'{(y_va==0).sum()} / {(y_va==1).sum()}':>22}")
print("(counts are on time / delayed)")

# Final training set for the hold-out evaluation: same procedure on the whole training set.
X_train_res, y_train_res = resampler.fit_resample(train[FEATURE_COLS], y_train_cls)
print(f"\nFull train after SMOTE: {np.bincount(y_train_res)}   (test set stays {np.bincount(y_test_cls)})")

fold           train before       train after SMOTE       val (untouched)
1               6435 / 2983             6435 / 6435           7409 / 2007
2              13844 / 4990           13844 / 13844           7887 / 1529
3              21731 / 6519           21731 / 21731           8061 / 1355
4              29792 / 7874           29792 / 29792           7246 / 2170
5             37038 / 10044           37038 / 37038           8040 / 1376
(counts are on time / delayed)

Full train after SMOTE: [45078 45078]   (test set stays [9743 4386])
